[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/01_orientation_and_precommit.ipynb)

# R2-Q2 Week 1 — Orient on the failure-mode literature and commit to your taxonomy

This notebook orients you on the failure-mode literature that R2-Q2 builds on — Adebayo et al. (2018) on saliency-method sanity checks, Noyan (2022) on PlantVillage background shortcuts, and the Grad-CAM mechanics you'll use to inspect misclassifications — and recaps the R2-Q1 outputs your work here inherits, including the headline finding that PlantVillage-trained accuracy on PlantDoc drops from 94.6% on fungal categories to 48.9% on healthy. The Week 1 deliverable is your `precommit.json` recording three written commitments: a five-category failure-mode taxonomy with decision rules, a categorization procedure, and the aggregation level you'll report at.

By the end of this notebook you will have:

- A `precommit.json` file recording your five-category taxonomy (with an "other / ambiguous" bucket), your categorization procedure, and your aggregation level — locked in writing before you see any Grad-CAM output.
- A documented understanding of the R2-Q1 outputs this question inherits: the trained classifier (`baseline_resnet18.pt`), the PD prediction table (`pd_predictions.parquet`, 236 rows), and the class-space mappings from R2-Q1's precommit.
- The literature context for Week 2's load-and-filter work and Week 3's Grad-CAM + sanity-check work: which methods pass Adebayo's checks, what shortcut learning looks like in PlantVillage specifically, and what "background-attended" failure modes have been observed in prior work.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive and set up irilab2026.
import irilab2026 as iri
iri.setup(gpu_required=False)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) What R2-Q2 inherits from R2-Q1

R2-Q1 measured the gap between lab and field conditions for plant disease classification. It trained a ResNet-18 classifier on PlantVillage — lab-quality, controlled-lighting, single-leaf-on-uniform-background images — and applied that trained model to PlantDoc, which is field photographs of the same diseases in real-world conditions: variable lighting, natural backgrounds, occluding objects, multiple leaves per image.

R2-Q1's headline finding: PlantVillage-trained accuracy on PlantDoc drops from 94.6% on fungal categories to 48.9% on healthy. The transfer failure isn't uniform — the model holds nearly all of its accuracy on one category and loses roughly half of it on another. *Where* the gap lives is settled. *What mechanism* produces it isn't.

That mechanism question is what R2-Q2 picks up. You'll run Grad-CAM on each PD misclassification, categorize the resulting attention heatmap into a pre-committed failure-mode taxonomy, and report the distribution across the five categories: background-attended, leaf-shape-attended, lighting-attended, symptom-attended-but-wrong-class, and other/ambiguous. The taxonomy itself, and the decision rules that operationalize it, are what you commit to in this notebook.

The R2-Q1 → R2-Q2 narrative bridge — *does the failure-mode distribution within healthy-misclassified images differ from the overall distribution?* — is held for the paper, where both questions' results can be discussed together. What's surfaced here is just the inheritance: R2-Q1 produced the predictions you'll inspect, the classifier you'll inspect them with, and the class-space mapping you'll use to translate model outputs into category labels.

### The four files you inherit

You don't load any of these in this notebook — N02 (Week 2) is where the loading happens. The inventory here is so you know what to expect when you open N02:

| File (in `iri.output_dir("r2_q1")`) | What it is | How N02–N04 use it |
|---|---|---|
| `baseline_resnet18.pt` | The trained PlantVillage classifier | N03 loads it for Grad-CAM passes and for the model-parameter randomization sanity check |
| `pd_predictions.parquet` | The 236-row PlantDoc prediction table | N02 filters to `correct_at_category == False`; N03 reads the model's predicted class per image as the Grad-CAM target |
| `precommit.json` | R2-Q1's Week 1 precommit | N02 pulls `pv_class_to_category` and `pd_class_to_category` for class-name lookups |
| `pv_predictions.parquet` | The PlantVillage-internal prediction table (reference) | Not strictly needed for R2-Q2; available if you want to cross-check PV-side predictions |

If you ran R2-Q1 yourself last week, these files are already in your `r2_q1` output directory. If you're starting R2-Q2 without having done R2-Q1, you'll need to run the R2-Q1 notebook series first — the R2-Q2 workflow assumes those files exist, and N02's first move is to load them.

## 3) The failure-mode literature — one paragraph per paper

Before you commit to your taxonomy in Section 4, three papers anchor what we know about the kind of failures you'll be inspecting and the tool you'll be inspecting them with. The treatment here is intentionally light — one paragraph per paper, just enough orientation to ground the pre-commit decisions in Sections 5–10. The papers are Adebayo et al. (2018) on whether attention-visualization methods can be trusted, Noyan (2022) on how PlantVillage models latch onto backgrounds, and Selvaraju et al. (2017) on the Grad-CAM mechanics themselves.

### 3.1) Adebayo et al. (2018) — Sanity checks for saliency maps

Saliency methods produce heatmaps over an input image showing which pixels contributed most to a model's prediction — they're the tool we use when we want to see what a model was "looking at." Adebayo et al. showed that several popular saliency methods produce visually plausible heatmaps even when applied to fully randomized models, meaning the heatmaps were reading patterns in the input image rather than what the model had learned. To separate real model attention from input-image patterns, they introduced two sanity checks: **model-parameter randomization** progressively replaces the trained model's weights with random values from the top layers backward and confirms that the saliency map diverges from the trained-model map as randomization proceeds; **data randomization** trains a model from scratch on label-shuffled data and confirms that the resulting saliency map differs from the real model's. A method that passes both is reading what the model actually learned. Vanilla Grad-CAM — the variant you'll use in N03 — passes both checks on most architectures Adebayo et al. tested, but that doesn't exempt you from running the checks on your specific trained model: Section 10 is where you commit to the sanity-check procedure, and N03 is where you execute it.

### 3.2) Noyan (2022) — Uncovering bias in the PlantVillage dataset

Noyan tested whether PlantVillage class labels could be predicted from the background alone — the regions of each image that aren't the leaf. By masking the leaves out and training classifiers on the remaining background pixels, Noyan found that PV labels remained predictable well above chance: even with the background downsampled to just eight pixels, classifiers reached around 49% accuracy on the 38-class task (chance would be roughly 3%). This is the empirical basis for the "background-attended" category in your taxonomy. If PV-trained models are learning background regularities that correlate with labels, those regularities are part of what the model uses to predict — and field photographs in PlantDoc don't share those regularities, which would explain a transfer failure concentrated in the conditions where the background changes most. Whether your specific trained classifier from R2-Q1 actually attended to backgrounds when it failed on PD is what Sections 4–8 of this notebook set you up to measure.

### 3.3) Selvaraju et al. (2017) — Grad-CAM

Selvaraju et al. introduced Gradient-weighted Class Activation Mapping (Grad-CAM): a method for producing a class-discriminative attention heatmap from any convolutional network by combining the gradients of a chosen class's score with the spatial activations of a chosen convolutional layer. Mechanically: take the gradient of the predicted-class score with respect to each feature map in the target layer, average those gradients over each map to get a per-map weight, take a weighted sum of the feature maps using those weights, apply ReLU to keep only positive contributions, and the result is a low-resolution heatmap showing which image regions most supported the chosen class. For ResNet-18 with 224×224 inputs and the last convolutional block (`layer4`) as the target, the heatmap comes out at 7×7 spatial resolution before being upsampled to the input size — coarse enough that "the model attended to the upper-left region" is the right grain of conclusion, not "the model attended to pixel (37, 84)." This is the tool you'll use in N03 to produce one heatmap per PD misclassification, and those heatmaps are the input that N04 categorizes against your pre-committed taxonomy.

## 4) Pre-commit your taxonomy

The Week-1 pre-commit discipline is straightforward: before you see any Grad-CAM output, you write down what you're going to do with the output. That includes the categories you'll sort failures into, the decision rules that operationalize each category, the aggregation level you'll report the distribution at, and the criterion your sanity checks have to meet. A taxonomy chosen after seeing the data is a taxonomy fit to the data — and the "the model fails in recognizable patterns" finding isn't a real test if the patterns were derived from the same images you're analyzing.

The four pre-commits become four top-level fields in `precommit.json`:

- **`taxonomy`** — the categories themselves (this section, Section 4).
- **`categorization_procedure`** — the rules for assigning each Grad-CAM heatmap to a category (Sections 5–8, one section per sub-decision).
- **`aggregation_level`** — what level you report the distribution at (Section 9).
- **`sanity_checks`** — the criterion your saliency method has to meet, per Adebayo (Section 10).

This section commits the first one and initializes the dict.

The taxonomy has five categories: four named, plus an "other / ambiguous" bucket. Each named category corresponds to a failure mode that prior work has either directly documented or that PlantVillage's data-collection conventions make plausible:

- **`background_attended`** — the model is looking at the background rather than the leaf. Grounded in Noyan (2022)'s finding that PV labels are predictable from the background alone at well above chance.
- **`leaf_shape_attended`** — the model is looking at the leaf outline rather than disease symptoms. PV's uniform single-leaf framing means leaf shape correlates with host species, and host species correlates with disease in PV's class structure — so leaf shape becomes a shortcut for disease identity.
- **`lighting_attended`** — the model is looking at lighting conditions. PV is studio-lit; PD has natural and variable lighting. If lighting co-varied with category in the training data, the model could be reading lighting cues that don't generalize to field photographs.
- **`symptom_attended_but_wrong_class`** — the model is looking at a real symptom on the leaf but assigning it to the wrong disease. This is the "honest mistake" category: the model engaged with the leaf and the lesion, but its class boundary was wrong.
- **`other_ambiguous`** — none of the above, or more than one of the above. The question page requires this category explicitly, as protection against forcing every failure into a prepared box. A non-trivial share of failures landing here is itself a result: it means the taxonomy as committed isn't capturing the actual structure of the failures, and that's a finding worth reporting in the paper.

The decision rules that operationalize these categories — the numeric thresholds, the predicate order, the leaf-segmentation method — don't live in this section. They get built up in Sections 5–8 under the `categorization_procedure` field. What you commit here is the *list* of categories; the *rules* are downstream.

In [ ]:
# Initialize the precommit dict with its four top-level fields.
# Each one will be filled in by the section indicated:
#   - taxonomy:                 this section (Section 4)
#   - categorization_procedure: Sections 5-8 (one sub-field per section)
#   - aggregation_level:        Section 9
#   - sanity_checks:            Section 10
precommit = {
    "taxonomy": None,
    "categorization_procedure": {},   # empty dict so Sections 5-8 can add sub-fields
    "aggregation_level": None,
    "sanity_checks": None,
}

# Commit the five-category taxonomy.
precommit["taxonomy"] = {
    "categories": [
        "background_attended",
        "leaf_shape_attended",
        "lighting_attended",
        "symptom_attended_but_wrong_class",
        "other_ambiguous",
    ],
    "decision_rules_pointer": "see categorization_procedure",
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: why these five categories (not four or six), how
        # the four named categories map to documented failure modes in prior
        # work, and what an unexpectedly large "other_ambiguous" share would
        # tell you about whether the taxonomy fits the actual failures.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
print(f"Taxonomy committed: {len(precommit['taxonomy']['categories'])} categories")
for cat in precommit["taxonomy"]["categories"]:
    print(f"  - {cat}")